# Step 1: Extract the data (24 conversations)
The NoXi zips are huge (about 14 GB). `tools/data24/organise.py` copies out only the 19 small files we need for each of the 24 conversations: the language label, and for the expert and the novice the engagement labels and the four streams (eGeMAPS voice, OpenFace 2.0 face, w2v-BERT 2.0 and VideoMAE embeddings, each as a header file and a data file).
*This notebook checks the result of that script on this computer. It does not read the zips; to redo the extraction run `python tools/data24/organise.py` (needs the NoXi zips).*

**In short** (what this notebook shows)

- **What:** Extracts the files that are needed (eGeMAPS voice, OpenFace face, engagement labels, language tag) from the NoXi zips into data/noxi/<session>/.
- **Needs the NoXi data?** Yes (the NoXi zips). The 24-conversation data was built with tools/data24/organise.py; this notebook shows the method. Old outputs were removed.
- **Result / use:** Use it to explain where the data comes from.

### Dataset
**NoXi** (NOvice eXpert Interaction) - expert-novice conversations with continuous engagement labels.
- Paper: A. Cafaro et al., ACM ICMI 2017 (https://doi.org/10.1145/3136755.3136780); challenge page: https://multimediate.perceptualui.org
- Used here: **24 conversations** (8 English, 8 French, 8 German; 48 participant recordings; 48,486 one-second windows).
- Licence: NoXi is available for research under an End-User Licence Agreement. The data is **not included** and must not be shared.

Go to the project folder.

In [2]:
import os
if os.getcwd().endswith("notebooks"):
    os.chdir("..")          # work from the project folder

**Imports.**

In [3]:
import pandas as pd

**The 24 conversations**, 8 per language, as chosen for the laptop and time limits (NoXi has 72 English, French and German conversations).

In [4]:
SESSIONS = {
    'English': ['026', '027', '028', '029', '030', '058', '062', '063'],
    'French':  ['001', '007', '009', '011', '013', '014', '020', '021'],
    'German':  ['068', '070', '072', '073', '074', '075', '076', '077'],
}
print({k: len(v) for k, v in SESSIONS.items()}, 'total', sum(len(v) for v in SESSIONS.values()))

{'English': 8, 'French': 8, 'German': 8} total 24


**The 19 files needed per conversation.**

In [5]:
STREAMS = ['audio.egemapsv2', 'openface2', 'audio.w2vbert2_embeddings', 'videomae']
NEEDED = ['language.annotation.csv']
for role in ('expert', 'novice'):
    NEEDED.append(f'{role}.engagement.annotation.csv')
    for s in STREAMS:
        NEEDED += [f'{role}.{s}.stream', f'{role}.{s}.stream~']     # header + data
print(len(NEEDED), 'files per conversation')
print(*NEEDED, sep='\n')

19 files per conversation
language.annotation.csv
expert.engagement.annotation.csv
expert.audio.egemapsv2.stream
expert.audio.egemapsv2.stream~
expert.openface2.stream
expert.openface2.stream~
expert.audio.w2vbert2_embeddings.stream
expert.audio.w2vbert2_embeddings.stream~
expert.videomae.stream
expert.videomae.stream~
novice.engagement.annotation.csv
novice.audio.egemapsv2.stream
novice.audio.egemapsv2.stream~
novice.openface2.stream
novice.openface2.stream~
novice.audio.w2vbert2_embeddings.stream
novice.audio.w2vbert2_embeddings.stream~
novice.videomae.stream
novice.videomae.stream~


**Check every conversation:** are all 19 files there, which language does the file say, and how big is the folder?

In [6]:
rows = []
for lang, sess in SESSIONS.items():
    for s in sess:
        folder = os.path.join('data', 'noxi', s)
        found = sum(os.path.exists(os.path.join(folder, f)) for f in NEEDED)
        in_file = open(os.path.join(folder, 'language.annotation.csv')).read().strip().split(';')[2]
        mb = sum(os.path.getsize(os.path.join(folder, f)) for f in NEEDED if os.path.exists(os.path.join(folder, f))) / 1e6
        rows.append(dict(session=s, language=lang, language_in_file=in_file, files_found=found, size_mb=round(mb)))
check = pd.DataFrame(rows)
print(check.to_string(index=False))
print()
print('all 19 files present:', bool((check.files_found == 19).all()))
print('file language matches our grouping:', bool((check.language == check.language_in_file).all()))
print('total size (MB):', int(check.size_mb.sum()))

session language language_in_file  files_found  size_mb
    026  English          English           19      468
    027  English          English           19      434
    028  English          English           19      756
    029  English          English           19      533
    030  English          English           19      423
    058  English          English           19      978
    062  English          English           19      878
    063  English          English           19      414
    001   French           French           19      690
    007   French           French           19     1407
    009   French           French           19      585
    011   French           French           19      932
    013   French           French           19      877
    014   French           French           19      692
    020   French           French           19      429
    021   French           French           19      560
    068   German           German           19  

**Redo the extraction** (needs the NoXi zips in the Downloads folder; set `RUN = True`). It skips files that are already copied.

In [7]:
RUN = False
if RUN:
    exec(open('tools/data24/organise.py').read())

**Result.** All 24 conversations are on disk with their 19 files. Next: step 2 turns them into one-second windows.